# W07b Colab B ·어텐션·블록·AE 기본 구조·웹 앱

생성형 인공지능 ·2026년10월14일 ·동덕여자대학교 데이터 사이언스 전공 ·유원상 교수

CPU에서 첫 셀부터 실행합니다. 두 노트북은 독립적입니다. GPU·유료API·개인파일 업로드·대형모델 다운로드가 필요 없습니다. 내장 작은 표와 합성 문자열을 사용하며 실제 LLM 출력이라고 표시하지 않습니다.

파일 → Drive에 사본 저장 후 사용하세요. 설치한 모듈을 이미 import했다면 런타임을 재시작하고 첫 셀부터 실행합니다. 실패 시 오류를 확인하며 가짜 결과로 대체하지 않습니다.

각 문제는 먼저 읽고 자신의 코드 셀에서 수정합니다. 자동 시연과 문제 답 변수를 분리했으므로 미완성 답이 있어도 시연 검사는 실행됩니다. 전체 실행 통과는 자신이 모든 문제를 풀었다는 뜻이 아닙니다.


## 공통 코드: 정의와 입력·출력

설치한 태그 `2026-fall-w07b`와 같은 정의 행입니다. 함수 이름을 클릭하여 구현을 바로 확인할 수 있습니다. 아래 API는 실습 편의를 위한 것이며, 연습문항은 기본 Python으로 풉니다.

### [softmax_scores](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L14)

유한 점수 N개 → 합1 확률 N개. T>0, 기본1; 입력은 수정하지 않는다.

최댓값을 뺀 뒤 T로 나누어 지수를 계산한다. 예: [0,log(2)] → [1/3,2/3].
빈 입력·비유한 값·양수가 아닌 T는 ValueError. 학습이나 추출은 하지 않는다.

### [filter_probs](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L31)

합1 비음수 확률 → {base, after_k, final, order, kept}.

top_k=0은 해제; 1..N 정수는 상위 개수. top_p는 (0,1], 기본1은 해제.
순서: top-k → 재정규화 → top-p → 재정규화. 경계를 넘는 토큰 포함.
동률은 원 인덱스 순서; final은 원 순서이며 제외 후보0. 입력 변경 없음.
예: filter_probs([.6,.3,.1],top_k=2)['final'] == [2/3,1/3,0].

### [attention_row](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L64)

Query1개·Key N×d·Value N×v → 점수/허용/가중치/출력 dict.

position은0-based Query 위치; causal=True이면 j<=position만 허용한다.
점수=q·k/sqrt(d), 허용점수에 softmax, 출력폭v의 Value 가중합.
학습·다운로드 없음. 잘못된 차원·비유한 값은 ValueError.
예: attention_row([1],[[0],[0]],[[2],[6]],1)['output'] == [4.0].

### [bigram_counts](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L92)

인접 문자 전이 횟수. 공백도 문자이며 끝→처음 연결 없음.

입력 길이2이상 문자열. 반환은 {출발문자:{다음문자:횟수}} 새 dict.
예: 'xyx' → {'x':{'y':1},'y':{'x':1}}. 파일을 읽지 않는다.

### [generate_one](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L107)

직전 문자 전이표에서1문자 선택. {context,next,table,mode} 반환.

seed는 로컬 Random에만 적용하여 전역 난수 상태를 바꾸지 않는다.
greedy 동률은 문자 정렬 순서. 매 호출 seed를 다시 설정하는 수업 실험이며,
긴 생성의 연속 난수 스트림을 모사한다고 주장하지 않는다. 미관측 문맥은 오류.

### [definition_link](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L132)

설치한 수업 함수의 실제 정의 시작 행 GitHub URL; 네트워크 요청 없음.

수업 패키지 함수만 허용한다. ref는 설치한 tag/commit과 같게 지정한다.

### [probability_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L9)

세 설정 → 표·설명·막대그래프. 가상 로짓[2.4,1.1,.2,-.6], 실제LLM 아님.

top_k 정수0..4, top_p(0,1], T>0. summary_fn(result)->str는 학생 연결 지점.
반환 (후보표 list, Markdown str, matplotlib Figure); 호출자는 Figure를 닫는다.

### [attention_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L32)

0-based 위치0..2와 마스크 → Key표·출력벡터·가중치 그래프.

Q=K=[[1,0],[0,1],[1,1]], V=[[2,1],[0,4],[3,-1]]은 명시한 수계산 예다.
(표 list, 설명 str, Figure) 반환; 학습·다운로드 없음.

### [generation_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L50)

문맥·정수seed·greedy → 새문맥·전이표·모드설명. 내장 합성 말뭉치 사용.

전이표 열은 다음문자/횟수/확률. 매 클릭1문자 추가. 실패는 명확한 오류로 전달.

### [build_review_app](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L59)

3패널 Gradio Blocks 생성, 서버는 시작하지 않음. CPU·API키/모델 다운로드 없음.

summary_fn(result)->str를 연결하면 확률 패널에 학생 설명을 표시한다.
예: app=build_review_app(); app.launch(share=True). 공유 URL은 런타임에 의존.


In [ ]:
import os, sys, subprocess, math
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_REF = '2026-fall-w07b'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        f'luna-genai @ git+https://github.com/lunalab-ai/genAI.git@{COURSE_REF}#subdirectory=src'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio==6.26.0', 'matplotlib>=3.8,<4'], check=True)
import luna_genai
assert luna_genai.__version__ == '0.1.12', '패키지 버전이 다릅니다. 설치 후 런타임을 재시작하세요.'
from luna_genai.review_core import softmax_scores, filter_probs, attention_row, bigram_counts, generate_one, definition_link
from luna_genai.review_app import probability_view, attention_view, generation_view, build_review_app
from IPython.display import display, Markdown
print('Python', sys.version.split()[0], '| course', luna_genai.__version__, COURSE_REF)


### 실제 소스 바로가기 재확인

아래 함수는 설치 파일에서 정의 줄을 찾아 같은 고정 태그 URL을 만듭니다. API 이름을 눌러 입력 검증과 중간 계산을 읽어 보세요.


In [ ]:
for fn in [softmax_scores, filter_probs, attention_row, bigram_counts, generate_one, probability_view, attention_view, generation_view, build_review_app, definition_link]:
    display(Markdown(f'[{fn.__name__}]({definition_link(fn, COURSE_REF)})'))


## 1. 설명 예제: 어텐션 한 행

Query=[0,0]이면두Key와의 점수가 모두0입니다. Value=[2,0],[4,6]의 가중 평균은[3,3]입니다. Query위치0을causal로 설정하면 첫Value만 허용됩니다. 아래 두 결과는 같은 표에서 마스크 조건만 바꿉니다.


In [ ]:
for pos in [0,1]:
    result = attention_row([0,0], [[1,0],[0,1]], [[2,0],[4,6]], pos, True)
    print(pos, result)


## 2. 표와 그림을 연결

앱의교육용Q/K/V는함수docstring에 모두 명시되어 있습니다. 표에는 원점수,허용여부,최종 가중치를 같이 표시합니다. 출력 특징 벡터의 합은1일 필요가 없습니다.


In [ ]:
import matplotlib.pyplot as plt
table, report, figure = attention_view(1, True)
print(table); print(report); display(figure); plt.close(figure)
print(generation_view('a', 7, True))


## 4. Python 연습

Notion·PDF와같은Q번호입니다. 빈칸/오류 코드는 설명 셀에 있으며 그대로 자동 실행하지 않습니다. 필요한 문법은 문제에 제공됩니다. 자신의 코드 셀에서 완성하고 출력뿐 아니라 이유도 기록하세요.


### Q64 ·내적 빈칸 (빈칸)

두 벡터의 내적이 되도록 빈칸을 채워라. 길이는 같다. 출력과 좌표별 곱도 제시하라.

```python
q = [2, -1, 1]
k = [1, 3, 2]
score = sum(___ for i in range(len(q)))
print(score)
```

목표: 어텐션의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q64_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q65 ·미래를 차단하는 조건 (디버깅)

0-based Query 위치 i =1에서 자기 위치까지 허용하려한다. 아래 조건을 수정하고 allowed를 써라. True는 참고 허용이다.

```python
i = 1
allowed = [j < i for j in range(3)]
print(allowed)
```

목표: 어텐션의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q65_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q66 ·가중합 코드 추적 (출력·해석)

코드 출력과 그 벡터의 길이를 구하라. weights의 합은 1이고 V의 각 행은 참고 위치 하나의 2 차원 특징이다.

```python
weights = [0.2, 0.8]
V = [[5, 1], [0, 6]]
out = [sum(weights[j] * V[j][d] for j in range(2)) for d in range(2)]
print(out)
```

목표: 어텐션의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q66_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q67 ·head 폭으로 스케일 (빈칸)

sqrt(x)는 양의 제곱근이다. 빈칸에 Key/Query 특징 수를 사용하여 scaled 점수를 구하라. score는 이미 내적 합이다.

```python
from math import sqrt
q = [1, 1, 1, 1]
score = 8
scaled = score / sqrt(___)
print(scaled)
```

목표: 어텐션의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q67_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q68 ·마스크 뒤 재정규화 (디버깅)

이미 구한 확률에서 미래 위치를 0으로 바꾸었다. 최종 허용 확률의 합을 1로 만드는 코드를 추가하라. 이 예에서는 허용 질량이 양수다.

```python
p = [0.2, 0.3, 0.5]
allowed = [True, True, False]
a = [p[j] if allowed[j] else 0 for j in range(3)]
print(a)
```

목표: 어텐션의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q68_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q74 ·잔차 덧셈 구현 (디버깅)

아래 코드는 입력 x와 변환 결과 fx의 잔 차 덧셈을 하려한다. 오류를 고치고 결과와 길이를 구하라. 두 리스트 길이는 같다.

```python
x = [1, -2, 3]
fx = [0.5, 1, -1]
y = x + fx
print(y)
```

목표: 블록·계열의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q74_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q75 ·위치별 FFN 코드 추적 (출력·해석)

아래 코드의 출력을 구하라. max(0,v)는 음 수를 0으로 바꾸는 ReLU이다. 서로 다른 행의 값을 직접 섞는지도 설명하라.

```python
X = [[-1, 2], [3, -2]]
Y = [[max(0, 2*v + 1) for v in row] for row in X]
print(Y)
```

목표: 블록·계열의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q75_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q76 ·LayerNorm의 평균 축 (디버깅)

각 행은 한 토큰 위치의 두 특징이다. 아래 코드는 모든 원소 평균을 빼고 있다. 각 행의 평균을 따로 빼도록 수정하고 결과를 써라. 이번 문제는 평균 제거 단계만 계산한다.

```python
X = [[1, 3], [10, 14]]
mu = sum(sum(row) for row in X) / 4
centered = [[v-mu for v in row] for row in X]
print(centered)
```

목표: 블록·계열의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q76_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q80 ·장난감 AE의 흐름 (빈칸)

다음 함수는 학습된 AE가 아닌 수계산용 역할 비유다. 빈칸을 채워 입력 x를 encode한 뒤 decode하라. z와 restored를 구하고, 원 입력과 완벽히 같은지도 설명하라.

```python
def encode(x):
    return [(x[0] + x[1]) / 2]
def decode(z):
    return [z[0], z[0]]
x = [2, 6]
z = ___
restored = ___
print(z, restored)
```

목표: AE 기초의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q80_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


## 5. 독립 실행 검사

이 검사는 설명 예제와 공통 모듈의 수치·마스크·callback을 확인합니다. 연습 문항의 정답을 공개하지 않으며,본인의 풀이를 자동 채점하지 않습니다.


In [ ]:
assert math.isclose(sum(softmax_scores([4,3,1])), 1)
assert softmax_scores([10000,10000]) == [0.5,0.5]
filtered = filter_probs([.55,.25,.20], top_p=.7)
assert filtered['kept'] == [0,1] and filtered['final'][2] == 0
assert math.isclose(sum(filtered['final']),1)
masked = attention_row([0,0], [[1,0],[0,1]], [[2,0],[4,6]],0,True)
assert masked['weights'] == [1.,0.] and masked['output']==[2.,0.]
full = attention_row([0,0], [[1,0],[0,1]], [[2,0],[4,6]],1,True)
assert full['output']==[3.,3.]
assert bigram_counts('xyxz') == {'x':{'y':1,'z':1},'y':{'x':1}}
assert generate_one('x',bigram_counts('xyxz'),11,True)['context']=='xy'
for callback,args in [(probability_view,(1.,2,.9)),(attention_view,(1,True))]:
    values=callback(*args)
    assert values[0] and values[1] and values[2] is not None
    plt.close(values[2])
assert len(generation_view('a',7,True)[0])==2
print('W07B-B RUNTIME CHECK PASSED')


## 마지막: 내가 설명하는 웹 애플리케이션

**APP01 · 요약 함수 연결**

아래 my_summary를 고쳐 temperature 설정과 남은 후보 수를 한 문장으로 출력하세요. result는 dict이며 result["temperature"]는 T, result["kept"]는 남은 인덱스 목록입니다. len(목록)은 개수입니다. 함수는 문자열을 반환합니다. 수정 후 app 생성·launch 셀을 다시 실행하세요.

목표: UI 설정 → 계산 결과 → 자신의 설명 함수의 연결을 이해합니다. 힌트: 문자열에 변수를 넣는 f-string을 사용할 수 있습니다. 자가 점검: 설정을 바꾸면 설명도 바뀌고 표의 남은 후보 수와 일치해야합니다.


In [ ]:
def my_summary(result):
    return '표에서 현재 설정과 남은 후보를 확인하세요.'
app = build_review_app(summary_fn=my_summary)


확률 패널에서 조건 하나만 바꾸어 표와 막대를 비교합니다. 어텐션 패널에서는 Query 위치와 미래 차단을 바꿉니다. 문자 생성 패널에서 문맥이 실제로 길어지는지 확인합니다. 모든 값은 교육용입니다. 공유 링크는 현재 런타임이 종료되면 사용할 수 없으므로 고정 노트북 URL에서 다시 실행하세요.


In [ ]:
if IN_COLAB:
    app.launch(share=True, debug=False, prevent_thread_lock=True)
else:
    print('Local: app built; actual browser validation is a separate check.')
